# 05 · Injury labels + linking to the club datasets

Project 1 (Injury Risk Modelling). The bucket holds a **curated injury dataset**
— dated injury spells with the cross-dataset identity mapping joined in.
Requires your key + `WASABI_USER` in `.env` (see `docs/data_access.md`).


In [ ]:
import nffc_data as nffc


## 1. Load the injuries + mapping file


In [ ]:
inj = nffc.load_parquet("injuries/gb1_injuries_with_mapping.parquet")
print(inj.shape)
inj[["player_name","statsbomb_id","second_spectrum_id","team_name","season","reason","from","until","days_missed"]].head()


## 2. Injury reasons (free text — needs normalising)
The `reason` column is raw text with inconsistent casing/synonyms; map it to
clean categories (e.g. flag soft-tissue injuries) before modelling.


In [ ]:
inj["reason"].value_counts().head(10)


## 3. Per-player injury burden
Aggregate spells into player-season features.


In [ ]:
burden = (inj.groupby(["player_name","season"])
            .agg(spells=("reason","size"), days_missed=("days_missed","sum"),
                 games_missed=("games_missed","sum"))
            .reset_index().sort_values("days_missed", ascending=False))
burden.head()


## 4. Linking injuries to the club datasets
The mapping is embedded in this file:
- `statsbomb_id` → StatsBomb events/lineups `player_id`
- `second_spectrum_id` → SecondSpectrum tracking player id (`ssiId`)

Once those datasets are uploaded, join on the matching id:


In [ ]:
print("injury players with a StatsBomb id:", inj["statsbomb_id"].notna().sum())
print("injury players with a SecondSpectrum id:", inj["second_spectrum_id"].notna().sum())

# matches = nffc.load_parquet("Statsbomb/Premier League/2023-2024/matches.parquet")
# events  = nffc.load_parquet("Statsbomb/Premier League/2023-2024/events/<match_id>.parquet")
# joined  = events.merge(inj, left_on="player_id", right_on="statsbomb_id", how="left")


Save your own engineered features under your personal folder:


In [ ]:
# feats = burden  # your features here
# nffc.upload_parquet(feats, "injury_burden.parquet")   # -> students/<you>/injury_burden.parquet
